In [ ]:
# 01 · EASY 집중 실험 CONFIG
CFG = {
    # GitHub 저장 · 기존 결과 보존, Easy 전용 Release
    'run_name': 'moveboxes_easy_lab_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'main',
    'output_root': '/content/moveboxes_runs',
    'source_run_name': 'moveboxes_stage_act_v1',
    'warm_start': True,

    # T4 / 설치 / 데이터
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 반복 예산 · 2,000회마다 실제 평가, 최대 20,000회
    'block_iters': 2000,
    'max_blocks': 10,
    'development_episodes': 8,
    'target_accuracy': 0.95,
    'success_streak': 2,
    'plateau_blocks': 4,

    # 작은 모델 · 실행 조건으로 학습, 첫 집기 구간 30% 별도 표집
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'amp': True,
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,
    'first_pick_fraction': 0.3,

    # 별도 시드와 동일한 200스텝 평가 제한
    'tuning_seed_start': 50000,
    'eval_seed_start': 70000,
    'final_episodes': 100,
    'test_seed_start': 40000,
    'test_episodes': 8,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},

    # 실행 / 출력
    'ensemble_candidates': [1, 4],
    'ensemble_window': 4,
    'temporal_decay': 0.25,
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'console_interval_seconds': 30,
    'team': 'my-team',

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'easy'/'code'))
for name in ('build_easy_notebook','easy_lab'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from build_easy_notebook import CONFIG as EASY_DEFAULTS
CFG = dict(EASY_DEFAULTS, **CFG)
from easy_lab import EasyLab, source_bundle
experiment = EasyLab(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증/복원
experiment.connect()
experiment.report()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · Easy GPU 확인 / 기존 성공 시연과 모델 가져오기
experiment.check_runtime()
experiment.prepare()


In [ ]:
# 06 · EASY 반복 학습 · 중단 후 같은 셀을 다시 실행하면 이어서 진행
history = experiment.run_blocks()


In [ ]:
# 07 · 최고 모델 8회 별도 테스트 + 영상 + 2회 행동 기록
experiment.test("easy")
experiment.diagnose("easy")


In [ ]:
# 08 · 학습 곡선과 최고 결과
experiment.report()


In [ ]:
# 09 · 목표 달성 모델의 별도 시드 최종 평가
experiment.final_evaluation()


In [ ]:
# 10 · 평가 완료한 Easy 모델만 패키징
experiment.package()
